In [2]:
import torch
import torch as nn
from torch.utils.data import Dataset,DataLoader


/home/suyog/anaconda3/envs/rsicd/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [15]:
import os
os.chdir("../")

In [ ]:
# Dataset class adapted from authors https://github.com/isaaccorley/torchrs 
import os
import json
from typing import List, Dict

import torch
import torchvision.transforms as T
from PIL import Image


class RSICD(torch.utils.data.Dataset):
    """ Image Captioning Dataset from 'Exploring Models and Data for
    Remote Sensing Image Caption Generation', Lu et al. (2017)
    https://arxiv.org/abs/1712.07835

    'RSICD is used for remote sensing image captioning task. more than ten thousands
    remote sensing images are collected from Google Earth, Baidu Map, MapABC, Tianditu.
    The images are fixed to 224X224 pixels with various resolutions. The total number of
    remote sensing images are 10921, with five sentences descriptions per image.'
    """
    splits = ["train", "val", "test"]

    def __init__(
        self,
        root: str = "data",
        split: str = "train",
        transform: T.Compose = T.Compose([T.ToTensor()])
    ):
        assert split in self.splits
        self.root = root
        self.transform = transform
        self.captions = self.load_captions(os.path.join(root, "dataset_rsicd.json"), split)
        self.image_root = "RSICD_images"

    @staticmethod
    def load_captions(path: str, split: str) -> List[Dict]:
        with open(path) as f:
            captions = json.load(f)["images"]
        return [c for c in captions if c["split"] == split]

    def __len__(self) -> int:
        return len(self.captions)

    def __getitem__(self, idx: int) -> Dict:
        captions = self.captions[idx]
        path = os.path.join(self.root, self.image_root, captions["filename"])
        x = Image.open(path).convert("RGB")
        x = self.transform(x)
        sentences = [sentence["raw"] for sentence in captions["sentences"]]
        return dict(x=x, captions=sentences)






Since our __getitem__ returns a image tensot with list of 5 raw captions(strings)

In [11]:
import os
print(os.getcwd())

/home/suyog/Projects/rsicd-image-captioning/notebooks


In [17]:
#first load the split
train_dataset = RSICD(root="data",split='train')


lets collect all the captions in a list inside our dataset

In [20]:
all_captions = []

for item in train_dataset:
    for captions in item['captions']:
        all_captions.append(captions)

In [25]:
len(all_captions),all_captions[5:10]

(43670,
 ['some planes are parked in an airport .',
  'the airport here is full of airplanes and containers .',
  'the airport here is full of airplanes and containers .',
  'some planes are parked in an airport .',
  'some planes are parked in an airport .'])

we have around 43 thousand captions in here

### Preprocessing
we have done some pretty basic preprocessing steps because the captions are fairly clean.

In [26]:
import re

def preprocess(captions: str) -> list:
    captions = captions.lower().strip()
    captions = re.sub(r"([?.!,])", r" \1 ", captions)# add space puntuation

    captions = re.sub(r"[^a-zA-Z?.!,]+", " ", captions)# remove unwanted characters

    captions = re.sub(r'\s+', ' ', captions).strip()

    tokens = captions.split()

    return tokens

In [28]:
tokenized_captions = []
for c in all_captions:
    tokenized_captions.append(preprocess(c))
    


In [30]:
tokenized_captions[1]

['many',
 'planes',
 'are',
 'parked',
 'next',
 'to',
 'a',
 'long',
 'building',
 'in',
 'an',
 'airport',
 '.']

We are using the simplest form of tokenization, wordpiece tokenization.

### Word frequencies
Lets see the word frequency of our captions

In [34]:
from collections import Counter

word_counter = Counter()

for tokens in tokenized_captions:
    word_counter.update(tokens) ## adds the count instead of replacing

In [37]:
print(f"The unique words present in captions {len(word_counter)} words")

The unique words present in captions 2326 words


In [38]:
word_counter.most_common(10)

[('.', 43670),
 ('a', 41743),
 ('are', 25698),
 ('green', 21129),
 ('many', 18873),
 ('in', 17515),
 ('trees', 17337),
 ('and', 16627),
 ('of', 15084),
 ('the', 14158)]

**Note**: We add '.' because the model could learn when the end of sentence is

Some words which appear less than a certain thershold like maybe 5 adds noise so it is good we remove them

In [40]:
min_freq = 2

vocab_words = [word for word, freq in word_counter.items() if freq >=min_freq]

print(f"Vocab size after applying the threshold : {len(vocab_words)}")

Vocab size after applying the threshold : 1717


Lets add special tokens like <PAD>, <SOS>,<EOS>,<UNK> in our vocab

 1) < PAD > - Used to make the sequence same
 2) < SOS > - Indicate start of sentence
 3) < EOS > - Indicate the end of sentence
 4) < UNK > - Replaces the rare words not found in vocab

In [41]:
special_tokens = ['<PAD>','<SOS>','<EOS>','<UNK>']

vocab = special_tokens + vocab_words

In [44]:
#create mappings

word2idx = {word: idx for idx,word in enumerate(vocab)}
idx2word = {idx: word for word,idx in word2idx.items()}






In [45]:
len(vocab)

1721

In [46]:
def caption_to_indices(tokens, word2idx):
    indices = [word2idx.get(word, word2idx['<UNK>']) for word in tokens]
    indices = [word2idx['<SOS>']] + indices + [word2idx['<EOS>']]
    return indices


In [ ]:
import torch
from torch import nn
from torchvision import models

efficient_netb0 = models.efficientnet_b0(pretrained=True)


class Encoder(nn.Module):
    def __init__(self,encoded_dim,projection_dim):
        super().__init__()
        self.encoder = nn.Sequential(*list(efficient_netb0.children())[:-1])
        self.fc = nn.Linear(encoded_dim,projection_dim) # project to same dim as decoder embedding


    def forward(self,images):
        features = self.encoder(images) #(B,1280,1,1)
        features = features.flatten(start_dim =1)
        #features = features.view(features.size(0),-1) # (B,1280)
        features = self.fc(features)

        return features


c:\Users\Acer nitro\anaconda3\envs\rsicd\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and will be removed in 0.15, please use 'weights' instead.
  warnings.warn(
c:\Users\Acer nitro\anaconda3\envs\rsicd\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and will be removed in 0.15. The current behavior is equivalent to passing `weights=EfficientNet_B0_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_B0_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [ ]:
import torch
from torch import nn

class Decoder(nn.Module):
    def __init__(self,embed_dim,hidden_dim,vocab_size,num_layers = 1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size,embed_dim)
        self.lstm = nn.LSTM(embed_dim,hidden_dim,num_layers,batch_first = True)
        self.fc = nn.Linear(hidden_dim,vocab_size)

    def forward(self, captions, features):
        # captions: (B, T)
        embeddings = self.embedding(captions)  # (B, T, embed_dim)
        embeddings = torch.cat((features.unsqueeze(1), embeddings), dim=1)  # prepend image feature
        outputs, _ = self.lstm(embeddings)  # (B, T+1, hidden_dim)
        outputs = self.fc(outputs)  # (B, T+1, vocab_size)
        return outputs

In [ ]:
class ImageCaptioningModel(nn.Module):
    def __init__(self,Encoder,Decoder):
        super().__init__()
        self.encoder = Encoder
        self.decoder = Decoder
        
    def forward(self,image,captions):
        features = self.encoder(image)
        output = self.decoder(features,captions)

In [ ]:
Encoded_dim = 1280 # this becuase of efficientNet
Projected_dim = 512

In [ ]:
encoder = Encoder(encoded_dim=Encoded_dim,projection_dim=Projected_dim)
decoder = Decoder(embed_dim=Projected_dim,hidden_dim=512,vocab_size=)

In [ ]:
base_model = ImageCaptioningModel(Encoder,Decoder)
EPOCHS = 50

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(base_model.parameters())